## Example 1: Basic point pattern

In [ ]:
import numpy as np
from pointpats import PointPattern, mean_center, std_distance

# Example 1: Basic point pattern
print("Example 1: Basic point pattern")

np.random.seed(42)
coords = np.random.uniform(0, 100, (50, 2))

# Create PointPattern
pp = PointPattern(coords)

print(f"Number of points: {pp.n}")
print(f"Hull area: {pp.hull_area:.2f}")
print(f"Mean center: {mean_center(coords)}")
print(f"Std distance: {std_distance(coords):.2f}")
print(f"Mean NND: {pp.mean_nnd:.2f}")

# Density
density = pp.n / pp.hull_area
print(f"Density: {density:.4f}")

# Cross-check the pattern statistics against direct calculations with numpy and scipy
from scipy.spatial import ConvexHull, cKDTree
assert pp.n == 50
assert np.isclose(pp.hull_area, ConvexHull(coords).volume)
assert 0 < pp.hull_area <= 100 * 100
assert np.allclose(mean_center(coords), coords.mean(axis=0))
assert np.isclose(std_distance(coords), np.sqrt(((coords - coords.mean(axis=0)) ** 2).sum(axis=1).mean()))
nn_dist, _ = cKDTree(coords).query(coords, k=2)
assert pp.mean_nnd > 0
assert np.isclose(pp.mean_nnd, nn_dist[:, 1].mean())

## Example 2: Distance statistics

In [ ]:
import numpy as np
from pointpats import PointPattern, g, f, k

# Example 2: Distance statistics
print("Example 2: Distance statistics")

np.random.seed(123)
coords = np.random.uniform(0, 100, (50, 2))
pp = PointPattern(coords)

# g function (nearest neighbor) - takes coordinates
support_g, g_vals = g(coords, support=30)
print(f"G function: {len(support_g)} bins")

# f function (empty space)
support_f, f_vals = f(coords, support=30)
print(f"F function: {len(support_f)} bins")

# k function (Ripley's K)
support_k, k_vals = k(coords, support=30)
print(f"K function: {len(support_k)} bins")

# PointPattern nearest neighbor
print(f"Mean NND: {pp.mean_nnd:.2f}")

# G and F are cumulative distributions (non-decreasing, between 0 and 1); K never decreases
assert len(support_g) == len(support_f) == len(support_k) == 30
assert len(g_vals) == len(f_vals) == len(k_vals) == 30
for name, vals in [("G", g_vals), ("F", f_vals)]:
    assert np.all((vals >= 0) & (vals <= 1)), name
    assert np.all(np.diff(vals) >= 0), name
assert g_vals[0] == 0 and g_vals[-1] == 1.0
assert np.all(np.diff(k_vals) >= 0) and k_vals[0] == 0
assert pp.n == 50 and pp.mean_nnd > 0

## Example 3: Random point patterns

In [ ]:
import numpy as np
from pointpats import PointPattern, random

# Example 3: Random point patterns
print("Example 3: Random patterns")

np.random.seed(456)

# Poisson pattern - hull is bounding box [xmin, ymin, xmax, ymax]
hull = np.array([0, 0, 100, 100])
poisson_coords = random.poisson(hull, size=60)
poisson_pp = PointPattern(poisson_coords)

print(f"Poisson points: {poisson_pp.n}")
print(f"Poisson hull area: {poisson_pp.hull_area:.2f}")

# Nearest neighbor comparison
poisson_nn = poisson_pp.mean_nnd
print(f"Poisson mean NND: {poisson_nn:.2f}")

# Uniform pattern
uniform_coords = np.random.uniform(0, 100, (60, 2))
uniform_pp = PointPattern(uniform_coords)
uniform_nn = uniform_pp.mean_nnd
print(f"Uniform mean NND: {uniform_nn:.2f}")

assert poisson_pp.n == 60 and uniform_pp.n == 60
assert poisson_coords.shape == (60, 2)
assert np.all((poisson_coords >= 0) & (poisson_coords <= 100))
assert 0 < poisson_pp.hull_area <= 100 * 100
assert poisson_nn > 0 and uniform_nn > 0

## Example 4: Clustering analysis

In [ ]:
import numpy as np
from pointpats import PointPattern

# Example 4: Clustering analysis
print("Example 4: Clustering analysis")

np.random.seed(789)

# Multiple clusters
cluster1 = np.random.normal([20, 20], 3, (15, 2))
cluster2 = np.random.normal([80, 80], 4, (20, 2))
background = np.random.uniform(0, 100, (10, 2))

coords = np.vstack([cluster1, cluster2, background])
pp = PointPattern(coords)

print(f"Total points: {pp.n}")

# k-nearest neighbor analysis: knn(k) returns (neighbor ids, distances), each n x k
mean_kth = []
for k in [1, 2, 3]:
    knn_ids, knn_d = pp.knn(k)
    assert knn_ids.shape == knn_d.shape == (pp.n, k)
    mean_kth.append(knn_d[:, -1].mean())
    print(f"{k}-NN distance: {mean_kth[-1]:.2f}")

print(f"Hull area: {pp.hull_area:.2f}")
print(f"Mean NND: {pp.mean_nnd:.2f}")

assert pp.n == 45
# The 1st-nearest-neighbor distance is the mean NND, and the k-th neighbor is never closer
assert np.isclose(mean_kth[0], pp.mean_nnd)
assert mean_kth[0] <= mean_kth[1] <= mean_kth[2]
assert pp.mean_nnd > 0

## Example 5: Custom boundary

In [ ]:
import numpy as np
from pointpats import PointPattern, g
from shapely.geometry import Point, Polygon

# Example 5: Custom boundary
print("Example 5: Custom boundary")

np.random.seed(999)

# Irregular polygon
boundary = Polygon([(10, 10), (90, 15), (85, 85), (15, 80)])

# Generate points in boundary
coords = []
while len(coords) < 40:
    x, y = np.random.uniform(10, 90, 2)
    if boundary.contains(Point(x, y)):
        coords.append([x, y])

coords_array = np.array(coords)
pp = PointPattern(coords_array)

print(f"Points: {pp.n}")
print(f"Boundary area: {boundary.area:.2f}")
print(f"Hull area: {pp.hull_area:.2f}")

# G function
support, g_vals = g(coords_array, support=20)
print(f"G function: {len(support)} bins")

assert pp.n == 40
assert all(boundary.contains(Point(x, y)) for x, y in coords_array)
# The boundary is convex, so the hull of points inside it cannot be larger than it
assert np.isclose(boundary.area, 5250.0)
assert 0 < pp.hull_area <= boundary.area
assert len(support) == len(g_vals) == 20
assert np.all(np.diff(g_vals) >= 0) and g_vals[-1] == 1.0